In [2]:
import numpy as np 
import pandas as pd 

In [3]:
df=pd.read_csv('../data/raw/city_hour.csv')

In [ ]:

print("Dataset shape:", df.shape)

exact_duplicates = df.duplicated()

print("\nExact duplicate rows:", exact_duplicates.sum())

print(
    "Percentage of dataset:",
    round(exact_duplicates.mean() * 100, 4),
    "%"
)

# Inspecting a few examples to check  if duplicates exist
if exact_duplicates.sum() > 0:
    display(df.loc[exact_duplicates].head(10))
else:
    print("\nNo exact duplicate rows found.")

Dataset shape: (707875, 16)

Exact duplicate rows: 0
Percentage of dataset: 0.0 %

No exact duplicate rows found.


In [ ]:
# Duplicating city-timestamp audit

# Counting rows sharing the same City and Datetime
duplicate_city_time = df.duplicated(
    subset=["City", "Datetime"],
    keep=False
)

print(
    "Rows involved in duplicate City-Datetime combinations:",
    duplicate_city_time.sum()
)

print(
    "Percentage of dataset:",
    round(duplicate_city_time.mean() * 100, 4),
    "%"
)

# Counting distinct duplicated City-Datetime combinations
duplicate_groups = (
    df.loc[duplicate_city_time, ["City", "Datetime"]]
    .drop_duplicates()
)

print(
    "\nNumber of duplicated City-Datetime combinations:",
    len(duplicate_groups)
)

# Displaying examples for investigation
if not duplicate_groups.empty:
    print("\nExample duplicate combinations:")
    
    example_keys = duplicate_groups.head(10)

    examples = df.merge(
        example_keys,
        on=["City", "Datetime"],
        how="inner"
    )

    display(examples.sort_values(["City", "Datetime"]))
else:
    print("\nNo duplicate City-Datetime combinations found.")

Rows involved in duplicate City-Datetime combinations: 0
Percentage of dataset: 0.0 %

Number of duplicated City-Datetime combinations: 0

No duplicate City-Datetime combinations found.


In [6]:
# Counting negative values in pollutant columns

pollutant_cols = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Xylene"
]

negative_counts = (
    df[pollutant_cols]
    .lt(0)
    .sum()
    .sort_values(ascending=False)
)

negative_summary = negative_counts.to_frame(
    name="Negative_Value_Count"
)

negative_summary["Percentage"] = (
    negative_summary["Negative_Value_Count"]
    / len(df)
    * 100
).round(4)

display(negative_summary)

,Negative_Value_Count,Percentage
PM2.5,0,0.0
PM10,0,0.0
NO,0,0.0
NO2,0,0.0
NOx,0,0.0
NH3,0,0.0
CO,0,0.0
SO2,0,0.0
O3,0,0.0
Benzene,0,0.0


In [ ]:
#  Investigating unusually high PM2.5 values

pm25 = df["PM2.5"]

print("PM2.5 observations:", pm25.notna().sum())
print("PM2.5 values equal to 999.99:", (pm25 == 999.99).sum())
print("PM2.5 values >= 900:", (pm25 >= 900).sum())
print("PM2.5 values >= 500:", (pm25 >= 500).sum())

print("\nHighest PM2.5 values:")
display(
    pm25.nlargest(15).to_frame(name="PM2.5")
)

PM2.5 observations: 562787
PM2.5 values equal to 999.99: 193
PM2.5 values >= 900: 500
PM2.5 values >= 500: 2120

Highest PM2.5 values:


,PM2.5
47509,999.99
74083,999.99
75293,999.99
75328,999.99
75329,999.99
75330,999.99
75331,999.99
75332,999.99
75333,999.99
75334,999.99


In [8]:
# Distribution of PM2.5 values equal to 999.99 by city

extreme_pm25 = df[df["PM2.5"] == 999.99]

print("Rows with PM2.5 = 999.99:", len(extreme_pm25))

display(
    extreme_pm25.groupby("City")
    .size()
    .sort_values(ascending=False)
    .to_frame(name="Count")
)

Rows with PM2.5 = 999.99: 193


,Count
City,
Gurugram,128
Amritsar,26
Lucknow,23
Bengaluru,8
Chennai,7
Ahmedabad,1


In [ ]:
# Inspecting records with unusually high PM2.5

extreme_pm25 = df[df["PM2.5"] == 999.99].copy()

print("Total rows with PM2.5 = 999.99:", len(extreme_pm25))

print("\nCounts by city:")
display(
    extreme_pm25.groupby("City")
    .size()
    .sort_values(ascending=False)
    .to_frame(name="Count")
)

print("\nSample records:")
display(
    extreme_pm25[
        [
            "City",
            "Datetime",
            "PM2.5",
            "PM10",
            "NO",
            "NO2",
            "NOx",
            "NH3",
            "CO",
            "SO2",
            "O3",
            "AQI",
            "AQI_Bucket",
        ]
    ].head(20)
)

Total rows with PM2.5 = 999.99: 193

Counts by city:


,Count
City,
Gurugram,128
Amritsar,26
Lucknow,23
Bengaluru,8
Chennai,7
Ahmedabad,1



Sample records:


,City,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,AQI,AQI_Bucket
47509,Ahmedabad,2020-06-02 14:00:00,999.99,NaN,1.44,36.25,18.6,NaN,0.3,18.56,NaN,226.0,Poor
74083,Amritsar,2017-03-17 07:00:00,999.99,84.24,19.24,28.49,NaN,12.23,0.0,2.10,15.78,310.0,Very Poor
75293,Amritsar,2017-05-06 17:00:00,999.99,110.21,11.61,16.73,NaN,29.76,0.0,1.47,15.61,122.0,Moderate
75328,Amritsar,2017-05-08 04:00:00,999.99,121.59,14.01,16.06,NaN,30.19,0.0,64.22,12.99,111.0,Moderate
75329,Amritsar,2017-05-08 05:00:00,999.99,114.84,13.60,16.03,NaN,30.36,0.0,42.39,14.48,110.0,Moderate
75330,Amritsar,2017-05-08 06:00:00,999.99,109.57,13.89,16.48,NaN,30.65,0.0,96.66,11.64,109.0,Moderate
75331,Amritsar,2017-05-08 07:00:00,999.99,113.44,14.70,16.49,NaN,30.79,0.0,72.33,12.06,108.0,Moderate
75332,Amritsar,2017-05-08 08:00:00,999.99,129.43,15.70,16.74,NaN,30.98,0.0,38.85,12.65,108.0,Moderate
75333,Amritsar,2017-05-08 09:00:00,999.99,134.00,15.66,16.45,NaN,30.43,0.0,14.71,13.56,109.0,Moderate
75334,Amritsar,2017-05-08 10:00:00,999.99,127.28,13.94,16.57,NaN,30.41,0.0,12.60,13.47,109.0,Moderate


In [ ]:
# Verifying the total number of PM2.5 values equal to 999.99

print("Total 999.99 values:", (df["PM2.5"] == 999.99).sum())

print("\nPM2.5 values at or above 900:")
print((df["PM2.5"] >= 900).sum())

print("\nUnique values at or above 900:")
display(
    df.loc[df["PM2.5"] >= 900, "PM2.5"]
    .value_counts()
    .sort_index()
    .to_frame(name="Count")
)

Total 999.99 values: 193

PM2.5 values at or above 900:
500

Unique values at or above 900:


,Count
PM2.5,
901.42,1
901.67,1
902.98,1
903.00,1
904.25,1
...,...
994.78,1
995.00,75
996.97,1


In [ ]:
# Investigating repeated extreme PM2.5 values

extreme_values = [999.99, 995.00]

for value in extreme_values:
    subset = df[df["PM2.5"] == value]

    print(f"\nPM2.5 = {value}")
    print("Total observations:", len(subset))

    print("\nObservations by city:")
    display(
        subset.groupby("City")
        .size()
        .sort_values(ascending=False)
        .to_frame(name="Count")
    )

    print("\nAQI_Bucket distribution:")
    display(
        subset["AQI_Bucket"]
        .value_counts(dropna=False)
        .to_frame(name="Count")
    )


PM2.5 = 999.99
Total observations: 193

Observations by city:


,Count
City,
Gurugram,128
Amritsar,26
Lucknow,23
Bengaluru,8
Chennai,7
Ahmedabad,1



AQI_Bucket distribution:


,Count
AQI_Bucket,
Severe,74
Very Poor,59
NaN,32
Moderate,13
Poor,12
Satisfactory,3



PM2.5 = 995.0
Total observations: 75

Observations by city:


,Count
City,
Hyderabad,75



AQI_Bucket distribution:


,Count
AQI_Bucket,
Severe,34
Very Poor,16
Poor,11
Moderate,10
Satisfactory,3
NaN,1


In [ ]:
# Inspecting hourly sequences of repeated extreme values

for city, value in [
    ("Gurugram", 999.99),
    ("Amritsar", 999.99),
    ("Hyderabad", 995.00),
]:
    city_df = (
        df.loc[df["City"] == city]
        .sort_values("Datetime")
        .copy()
    )

    city_df["is_target_value"] = city_df["PM2.5"].eq(value)

    # Identifying transitions into and out of sequences
    city_df["sequence_id"] = (
        city_df["is_target_value"]
        .ne(city_df["is_target_value"].shift())
        .cumsum()
    )

    sequences = (
        city_df.loc[city_df["is_target_value"]]
        .groupby("sequence_id")
        .agg(
            start_time=("Datetime", "min"),
            end_time=("Datetime", "max"),
            observations=("Datetime", "size"),
        )
        .reset_index(drop=True)
    )

    print(f"\nCity: {city} | PM2.5 = {value}")
    print("Number of separate sequences:", len(sequences))

    if not sequences.empty:
        print("\nLongest sequences:")
        display(
            sequences.sort_values(
                "observations",
                ascending=False
            ).head(10)
        )


City: Gurugram | PM2.5 = 999.99
Number of separate sequences: 31

Longest sequences:


,start_time,end_time,observations
28,2017-08-29 20:00:00,2017-08-30 11:00:00,16
2,2016-03-09 20:00:00,2016-03-10 03:00:00,8
20,2017-07-09 00:00:00,2017-07-09 06:00:00,7
19,2017-07-07 08:00:00,2017-07-07 13:00:00,6
23,2017-07-14 01:00:00,2017-07-14 06:00:00,6
11,2017-06-07 02:00:00,2017-06-07 07:00:00,6
9,2017-05-26 02:00:00,2017-05-26 06:00:00,5
27,2017-07-30 00:00:00,2017-07-30 04:00:00,5
18,2017-07-05 14:00:00,2017-07-05 18:00:00,5
15,2017-06-18 15:00:00,2017-06-18 18:00:00,4



City: Amritsar | PM2.5 = 999.99
Number of separate sequences: 10

Longest sequences:


,start_time,end_time,observations
2,2017-05-08 04:00:00,2017-05-08 13:00:00,10
4,2017-05-10 22:00:00,2017-05-11 00:00:00,3
5,2017-05-11 05:00:00,2017-05-11 07:00:00,3
7,2017-05-24 21:00:00,2017-05-24 22:00:00,2
6,2017-05-11 09:00:00,2017-05-11 10:00:00,2
3,2017-05-10 15:00:00,2017-05-10 16:00:00,2
1,2017-05-06 17:00:00,2017-05-06 17:00:00,1
0,2017-03-17 07:00:00,2017-03-17 07:00:00,1
8,2017-09-30 13:00:00,2017-09-30 13:00:00,1
9,2017-10-31 01:00:00,2017-10-31 01:00:00,1



City: Hyderabad | PM2.5 = 995.0
Number of separate sequences: 32

Longest sequences:


,start_time,end_time,observations
13,2015-06-24 06:00:00,2015-06-24 16:00:00,11
26,2016-06-16 02:00:00,2016-06-16 11:00:00,10
7,2015-06-01 19:00:00,2015-06-02 02:00:00,8
8,2015-06-02 04:00:00,2015-06-02 10:00:00,7
24,2016-06-15 14:00:00,2016-06-15 17:00:00,4
25,2016-06-15 21:00:00,2016-06-16 00:00:00,4
16,2015-08-21 13:00:00,2015-08-21 15:00:00,3
1,2015-04-14 01:00:00,2015-04-14 02:00:00,2
15,2015-08-20 13:00:00,2015-08-20 14:00:00,2
21,2015-09-04 15:00:00,2015-09-04 16:00:00,2


In [13]:
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")

print("Datetime dtype:", df["Datetime"].dtype)
print("Missing timestamps after conversion:", df["Datetime"].isna().sum())

Datetime dtype: datetime64[us]
Missing timestamps after conversion: 0


In [ ]:
#  Verifying hourly continuity in the longest sequences

cases = [
    ("Gurugram", 999.99, "2017-08-29 20:00:00", "2017-08-30 11:00:00"),
    ("Amritsar", 999.99, "2017-05-08 04:00:00", "2017-05-08 13:00:00"),
    ("Hyderabad", 995.00, "2015-06-24 06:00:00", "2015-06-24 16:00:00"),
]

for city, value, start, end in cases:
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    sequence = (
        df.loc[
            (df["City"] == city)
            & (df["PM2.5"] == value)
            & (df["Datetime"].between(start, end)),
            ["City", "Datetime", "PM2.5", "PM10", "AQI", "AQI_Bucket"]
        ]
        .sort_values("Datetime")
        .copy()
    )

    sequence["hours_since_previous"] = (
        sequence["Datetime"].diff().dt.total_seconds() / 3600
    )

    print(f"\n{city} | PM2.5 = {value}")
    print("Rows in interval:", len(sequence))
    print(
        "All timestamp gaps exactly one hour:",
        sequence["hours_since_previous"].dropna().eq(1).all()
    )

    display(sequence)


Gurugram | PM2.5 = 999.99
Rows in interval: 16
All timestamp gaps exactly one hour: True


,City,Datetime,PM2.5,PM10,AQI,AQI_Bucket,hours_since_previous
312605,Gurugram,2017-08-29 20:00:00,999.99,NaN,NaN,NaN,NaN
312606,Gurugram,2017-08-29 21:00:00,999.99,NaN,NaN,NaN,1.0
312607,Gurugram,2017-08-29 22:00:00,999.99,NaN,NaN,NaN,1.0
312608,Gurugram,2017-08-29 23:00:00,999.99,NaN,NaN,NaN,1.0
312609,Gurugram,2017-08-30 00:00:00,999.99,NaN,NaN,NaN,1.0
312610,Gurugram,2017-08-30 01:00:00,999.99,NaN,NaN,NaN,1.0
312611,Gurugram,2017-08-30 02:00:00,999.99,NaN,NaN,NaN,1.0
312612,Gurugram,2017-08-30 03:00:00,999.99,NaN,NaN,NaN,1.0
312613,Gurugram,2017-08-30 04:00:00,999.99,NaN,NaN,NaN,1.0
312614,Gurugram,2017-08-30 05:00:00,999.99,NaN,NaN,NaN,1.0



Amritsar | PM2.5 = 999.99
Rows in interval: 10
All timestamp gaps exactly one hour: True


,City,Datetime,PM2.5,PM10,AQI,AQI_Bucket,hours_since_previous
75328,Amritsar,2017-05-08 04:00:00,999.99,121.59,111.0,Moderate,NaN
75329,Amritsar,2017-05-08 05:00:00,999.99,114.84,110.0,Moderate,1.0
75330,Amritsar,2017-05-08 06:00:00,999.99,109.57,109.0,Moderate,1.0
75331,Amritsar,2017-05-08 07:00:00,999.99,113.44,108.0,Moderate,1.0
75332,Amritsar,2017-05-08 08:00:00,999.99,129.43,108.0,Moderate,1.0
75333,Amritsar,2017-05-08 09:00:00,999.99,134.00,109.0,Moderate,1.0
75334,Amritsar,2017-05-08 10:00:00,999.99,127.28,109.0,Moderate,1.0
75335,Amritsar,2017-05-08 11:00:00,999.99,114.23,109.0,Moderate,1.0
75336,Amritsar,2017-05-08 12:00:00,999.99,100.20,108.0,Moderate,1.0
75337,Amritsar,2017-05-08 13:00:00,999.99,89.00,108.0,Moderate,1.0



Hyderabad | PM2.5 = 995.0
Rows in interval: 11
All timestamp gaps exactly one hour: True


,City,Datetime,PM2.5,PM10,AQI,AQI_Bucket,hours_since_previous
353572,Hyderabad,2015-06-24 06:00:00,995.0,NaN,88.0,Satisfactory,NaN
353573,Hyderabad,2015-06-24 07:00:00,995.0,NaN,213.0,Poor,1.0
353574,Hyderabad,2015-06-24 08:00:00,995.0,NaN,311.0,Very Poor,1.0
353575,Hyderabad,2015-06-24 09:00:00,995.0,NaN,343.0,Very Poor,1.0
353576,Hyderabad,2015-06-24 10:00:00,995.0,NaN,374.0,Very Poor,1.0
353577,Hyderabad,2015-06-24 11:00:00,995.0,NaN,406.0,Severe,1.0
353578,Hyderabad,2015-06-24 12:00:00,995.0,NaN,437.0,Severe,1.0
353579,Hyderabad,2015-06-24 13:00:00,995.0,NaN,469.0,Severe,1.0
353580,Hyderabad,2015-06-24 14:00:00,995.0,NaN,500.0,Severe,1.0
353581,Hyderabad,2015-06-24 15:00:00,995.0,NaN,532.0,Severe,1.0


In [ ]:
#  comparing other pollutant measurements
# when PM2.5 equals 999.99 versus ordinary PM2.5 values.

pollutant_columns = [
    "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3",
    "Benzene", "Toluene", "Xylene"
]

suspicious = df["PM2.5"].eq(999.99)
ordinary = df["PM2.5"].notna() & ~suspicious

comparison = pd.DataFrame({
    "999.99 readings": df.loc[suspicious, pollutant_columns].mean(),
    "Other non-missing PM2.5 readings": df.loc[ordinary, pollutant_columns].mean()
})

comparison["999.99 count"] = (
    df.loc[suspicious, pollutant_columns].count()
)

comparison["Other readings count"] = (
    df.loc[ordinary, pollutant_columns].count()
)

display(comparison.round(2))

,999.99 readings,Other non-missing PM2.5 readings,999.99 count,Other readings count
PM2.5,999.99,67.30,193,562594
PM10,147.38,119.49,26,390441
NO,12.28,17.76,188,549642
NO2,20.74,29.81,181,548267
NOx,21.06,32.65,49,526504
NH3,34.05,24.37,37,405423
CO,0.59,2.34,175,541495
SO2,7.94,13.80,184,539832
O3,23.47,35.42,180,538526
Benzene,0.43,3.30,145,485515


In [ ]:

# Making sure Datetime is in datetime format
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")

# Inspecting the hours surrounding the suspicious readings
mask = (
    (df["City"] == "Amritsar") &
    (df["Datetime"] >= "2017-05-07 00:00:00") &
    (df["Datetime"] <= "2017-05-09 23:00:00")
)

df.loc[mask, ["City", "Datetime", "PM2.5", "PM10", "AQI", "AQI_Bucket"]]


,City,Datetime,PM2.5,PM10,AQI,AQI_Bucket
75300,Amritsar,2017-05-07 00:00:00,NaN,NaN,124.0,Moderate
75301,Amritsar,2017-05-07 01:00:00,NaN,188.16,124.0,Moderate
75302,Amritsar,2017-05-07 02:00:00,NaN,183.86,125.0,Moderate
75303,Amritsar,2017-05-07 03:00:00,NaN,182.84,125.0,Moderate
75304,Amritsar,2017-05-07 04:00:00,NaN,182.47,125.0,Moderate
...,...,...,...,...,...,...
75367,Amritsar,2017-05-09 19:00:00,NaN,NaN,138.0,Moderate
75368,Amritsar,2017-05-09 20:00:00,NaN,NaN,NaN,NaN
75369,Amritsar,2017-05-09 21:00:00,NaN,NaN,NaN,NaN
75370,Amritsar,2017-05-09 22:00:00,NaN,NaN,NaN,NaN


In [17]:

mask = (
    (df["City"] == "Amritsar") &
    (df["Datetime"] >= "2017-05-08 00:00:00") &
    (df["Datetime"] <= "2017-05-09 00:00:00")
)

print(
    df.loc[mask, ["Datetime", "PM2.5", "PM10", "AQI", "AQI_Bucket"]]
    .to_string(index=False)
)



           Datetime  PM2.5   PM10   AQI AQI_Bucket
2017-05-08 00:00:00  86.13 144.19 117.0   Moderate
2017-05-08 01:00:00    NaN 138.38 116.0   Moderate
2017-05-08 02:00:00    NaN 133.50 115.0   Moderate
2017-05-08 03:00:00 337.85 127.11 113.0   Moderate
2017-05-08 04:00:00 999.99 121.59 111.0   Moderate
2017-05-08 05:00:00 999.99 114.84 110.0   Moderate
2017-05-08 06:00:00 999.99 109.57 109.0   Moderate
2017-05-08 07:00:00 999.99 113.44 108.0   Moderate
2017-05-08 08:00:00 999.99 129.43 108.0   Moderate
2017-05-08 09:00:00 999.99 134.00 109.0   Moderate
2017-05-08 10:00:00 999.99 127.28 109.0   Moderate
2017-05-08 11:00:00 999.99 114.23 109.0   Moderate
2017-05-08 12:00:00 999.99 100.20 108.0   Moderate
2017-05-08 13:00:00 999.99  89.00 108.0   Moderate
2017-05-08 14:00:00    NaN  76.34 108.0   Moderate
2017-05-08 15:00:00    NaN  68.01 107.0   Moderate
2017-05-08 16:00:00    NaN  67.28 107.0   Moderate
2017-05-08 17:00:00    NaN  73.54 107.0   Moderate
2017-05-08 18:00:00    NaN  85.

In [ ]:

# Counting occurrences of suspicious PM2.5 values by city
suspicious = df[df["PM2.5"].isin([999.99, 995.00])]

print("Total suspicious rows:", len(suspicious))

print("\nOccurrences by city:")
print(
    suspicious.groupby(["City", "PM2.5"])
    .size()
    .sort_values(ascending=False)
)

print("\nAQI categories for these rows:")
print(suspicious.groupby(["PM2.5", "AQI_Bucket"], dropna=False).size())




Total suspicious rows: 268

Occurrences by city:
City       PM2.5 
Gurugram   999.99    128
Hyderabad  995.00     75
Amritsar   999.99     26
Lucknow    999.99     23
Bengaluru  999.99      8
Chennai    999.99      7
Ahmedabad  999.99      1
dtype: int64

AQI categories for these rows:
PM2.5   AQI_Bucket  
995.00  Moderate        10
        Poor            11
        Satisfactory     3
        Severe          34
        Very Poor       16
        NaN              1
999.99  Moderate        13
        Poor            12
        Satisfactory     3
        Severe          74
        Very Poor       59
        NaN             32
dtype: int64


In [ ]:
# Converting timestamps to datetime
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")

# Inspect suspicious values in chronological order
suspicious = df[df["PM2.5"].isin([999.99, 995.00])]

print(
    suspicious[["City", "Datetime", "PM2.5", "AQI", "AQI_Bucket"]]
    .sort_values(["City", "Datetime"])
    .to_string(index=False)
)

     City            Datetime  PM2.5   AQI   AQI_Bucket
Ahmedabad 2020-06-02 14:00:00 999.99 226.0         Poor
 Amritsar 2017-03-17 07:00:00 999.99 310.0    Very Poor
 Amritsar 2017-05-06 17:00:00 999.99 122.0     Moderate
 Amritsar 2017-05-08 04:00:00 999.99 111.0     Moderate
 Amritsar 2017-05-08 05:00:00 999.99 110.0     Moderate
 Amritsar 2017-05-08 06:00:00 999.99 109.0     Moderate
 Amritsar 2017-05-08 07:00:00 999.99 108.0     Moderate
 Amritsar 2017-05-08 08:00:00 999.99 108.0     Moderate
 Amritsar 2017-05-08 09:00:00 999.99 109.0     Moderate
 Amritsar 2017-05-08 10:00:00 999.99 109.0     Moderate
 Amritsar 2017-05-08 11:00:00 999.99 109.0     Moderate
 Amritsar 2017-05-08 12:00:00 999.99 108.0     Moderate
 Amritsar 2017-05-08 13:00:00 999.99 108.0     Moderate
 Amritsar 2017-05-10 15:00:00 999.99   NaN          NaN
 Amritsar 2017-05-10 16:00:00 999.99   NaN          NaN
 Amritsar 2017-05-10 22:00:00 999.99   NaN          NaN
 Amritsar 2017-05-10 23:00:00 999.99   NaN      

In [20]:
# Ensuring chronological order
df = df.sort_values(["City", "Datetime"]).copy()

# Calculating a 24-hour rolling average, requiring at least 16 readings
df["PM2.5_24hr_avg"] = (
    df.groupby("City")["PM2.5"]
    .transform(lambda s: s.rolling(window=24, min_periods=16).mean())
)

# Inspecting the Amritsar sequence
mask = (
    (df["City"] == "Amritsar") &
    (df["Datetime"] >= "2017-05-08 00:00:00") &
    (df["Datetime"] <= "2017-05-09 00:00:00")
)

print(
    df.loc[mask, ["Datetime", "PM2.5", "PM2.5_24hr_avg", "AQI", "AQI_Bucket"]]
    .to_string(index=False)
)

           Datetime  PM2.5  PM2.5_24hr_avg   AQI AQI_Bucket
2017-05-08 00:00:00  86.13             NaN 117.0   Moderate
2017-05-08 01:00:00    NaN             NaN 116.0   Moderate
2017-05-08 02:00:00    NaN             NaN 115.0   Moderate
2017-05-08 03:00:00 337.85             NaN 113.0   Moderate
2017-05-08 04:00:00 999.99             NaN 111.0   Moderate
2017-05-08 05:00:00 999.99             NaN 110.0   Moderate
2017-05-08 06:00:00 999.99             NaN 109.0   Moderate
2017-05-08 07:00:00 999.99             NaN 108.0   Moderate
2017-05-08 08:00:00 999.99             NaN 108.0   Moderate
2017-05-08 09:00:00 999.99             NaN 109.0   Moderate
2017-05-08 10:00:00 999.99             NaN 109.0   Moderate
2017-05-08 11:00:00 999.99             NaN 109.0   Moderate
2017-05-08 12:00:00 999.99             NaN 108.0   Moderate
2017-05-08 13:00:00 999.99             NaN 108.0   Moderate
2017-05-08 14:00:00    NaN             NaN 108.0   Moderate
2017-05-08 15:00:00    NaN             N

In [21]:
city = "Amritsar"
start = "2017-05-08 00:00:00"
end = "2017-05-09 00:00:00"

mask = (
    (df["City"] == city) &
    (df["Datetime"] >= start) &
    (df["Datetime"] <= end)
)

cols = [
    "City", "Datetime", "PM2.5", "PM10",
    "AQI", "AQI_Bucket"
]

print(df.loc[mask, cols].to_string(index=False))

    City            Datetime  PM2.5   PM10   AQI AQI_Bucket
Amritsar 2017-05-08 00:00:00  86.13 144.19 117.0   Moderate
Amritsar 2017-05-08 01:00:00    NaN 138.38 116.0   Moderate
Amritsar 2017-05-08 02:00:00    NaN 133.50 115.0   Moderate
Amritsar 2017-05-08 03:00:00 337.85 127.11 113.0   Moderate
Amritsar 2017-05-08 04:00:00 999.99 121.59 111.0   Moderate
Amritsar 2017-05-08 05:00:00 999.99 114.84 110.0   Moderate
Amritsar 2017-05-08 06:00:00 999.99 109.57 109.0   Moderate
Amritsar 2017-05-08 07:00:00 999.99 113.44 108.0   Moderate
Amritsar 2017-05-08 08:00:00 999.99 129.43 108.0   Moderate
Amritsar 2017-05-08 09:00:00 999.99 134.00 109.0   Moderate
Amritsar 2017-05-08 10:00:00 999.99 127.28 109.0   Moderate
Amritsar 2017-05-08 11:00:00 999.99 114.23 109.0   Moderate
Amritsar 2017-05-08 12:00:00 999.99 100.20 108.0   Moderate
Amritsar 2017-05-08 13:00:00 999.99  89.00 108.0   Moderate
Amritsar 2017-05-08 14:00:00    NaN  76.34 108.0   Moderate
Amritsar 2017-05-08 15:00:00    NaN  68.

In [22]:
city = "Amritsar"

mask = (
    (df["City"] == city) &
    (df["Datetime"] >= "2017-05-08 04:00:00") &
    (df["Datetime"] <= "2017-05-08 13:00:00")
)

pollutants = [
    "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3",
    "AQI", "AQI_Bucket"
]

print(df.loc[mask, ["Datetime"] + pollutants].to_string(index=False))

           Datetime  PM2.5   PM10    NO   NO2  NOx   NH3  CO   SO2    O3   AQI AQI_Bucket
2017-05-08 04:00:00 999.99 121.59 14.01 16.06  NaN 30.19 0.0 64.22 12.99 111.0   Moderate
2017-05-08 05:00:00 999.99 114.84 13.60 16.03  NaN 30.36 0.0 42.39 14.48 110.0   Moderate
2017-05-08 06:00:00 999.99 109.57 13.89 16.48  NaN 30.65 0.0 96.66 11.64 109.0   Moderate
2017-05-08 07:00:00 999.99 113.44 14.70 16.49  NaN 30.79 0.0 72.33 12.06 108.0   Moderate
2017-05-08 08:00:00 999.99 129.43 15.70 16.74  NaN 30.98 0.0 38.85 12.65 108.0   Moderate
2017-05-08 09:00:00 999.99 134.00 15.66 16.45  NaN 30.43 0.0 14.71 13.56 109.0   Moderate
2017-05-08 10:00:00 999.99 127.28 13.94 16.57  NaN 30.41 0.0 12.60 13.47 109.0   Moderate
2017-05-08 11:00:00 999.99 114.23 12.72 16.01  NaN 30.48 0.0 12.56 14.64 109.0   Moderate
2017-05-08 12:00:00 999.99 100.20 12.03 15.55  NaN 30.57 0.0  8.05 15.24 108.0   Moderate
2017-05-08 13:00:00 999.99  89.00 12.59 15.80  NaN 30.34 0.0  7.00 15.71 108.0   Moderate


In [23]:
pollutants = [
    "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3",
    "Benzene", "Toluene", "Xylene"
]

for col in pollutants:
    exact_999 = (df[col] == 999.99).sum()
    exact_995 = (df[col] == 995.00).sum()

    if exact_999 > 0 or exact_995 > 0:
        print(
            f"{col}: "
            f"999.99 = {exact_999}, "
            f"995.00 = {exact_995}"
        )

PM2.5: 999.99 = 193, 995.00 = 75
PM10: 999.99 = 41, 995.00 = 0


In [24]:
df = df.sort_values(["City", "Datetime"]).copy()

df["PM2.5_24hr_count"] = (
    df.groupby("City")["PM2.5"]
    .transform(lambda s: s.rolling(window=24, min_periods=1).count())
)

mask = (
    (df["City"] == "Amritsar") &
    (df["Datetime"] >= "2017-05-08 00:00:00") &
    (df["Datetime"] <= "2017-05-09 00:00:00")
)

print(
    df.loc[
        mask,
        ["Datetime", "PM2.5", "PM2.5_24hr_count", "PM2.5_24hr_avg"]
    ].to_string(index=False)
)

           Datetime  PM2.5  PM2.5_24hr_count  PM2.5_24hr_avg
2017-05-08 00:00:00  86.13               4.0             NaN
2017-05-08 01:00:00    NaN               4.0             NaN
2017-05-08 02:00:00    NaN               4.0             NaN
2017-05-08 03:00:00 337.85               5.0             NaN
2017-05-08 04:00:00 999.99               6.0             NaN
2017-05-08 05:00:00 999.99               7.0             NaN
2017-05-08 06:00:00 999.99               8.0             NaN
2017-05-08 07:00:00 999.99               9.0             NaN
2017-05-08 08:00:00 999.99              10.0             NaN
2017-05-08 09:00:00 999.99              11.0             NaN
2017-05-08 10:00:00 999.99              12.0             NaN
2017-05-08 11:00:00 999.99              13.0             NaN
2017-05-08 12:00:00 999.99              14.0             NaN
2017-05-08 13:00:00 999.99              15.0             NaN
2017-05-08 14:00:00    NaN              15.0             NaN
2017-05-08 15:00:00    N

In [25]:
mask = (
    (df["City"] == "Amritsar") &
    (df["Datetime"] >= "2017-05-07 00:00:00") &
    (df["Datetime"] <= "2017-05-08 03:00:00")
)

print(
    df.loc[
        mask,
        ["Datetime", "PM2.5", "PM2.5_24hr_count", "PM2.5_24hr_avg"]
    ].to_string(index=False)
)

           Datetime  PM2.5  PM2.5_24hr_count  PM2.5_24hr_avg
2017-05-07 00:00:00    NaN              11.0             NaN
2017-05-07 01:00:00    NaN              11.0             NaN
2017-05-07 02:00:00    NaN              11.0             NaN
2017-05-07 03:00:00    NaN              10.0             NaN
2017-05-07 04:00:00    NaN               9.0             NaN
2017-05-07 05:00:00    NaN               8.0             NaN
2017-05-07 06:00:00    NaN               7.0             NaN
2017-05-07 07:00:00    NaN               6.0             NaN
2017-05-07 08:00:00    NaN               5.0             NaN
2017-05-07 09:00:00    NaN               4.0             NaN
2017-05-07 10:00:00    NaN               4.0             NaN
2017-05-07 11:00:00    NaN               4.0             NaN
2017-05-07 12:00:00    NaN               4.0             NaN
2017-05-07 13:00:00    NaN               4.0             NaN
2017-05-07 14:00:00    NaN               4.0             NaN
2017-05-07 15:00:00    N